In [1]:
%%file loopback_arb.py

from acadia.runtime import Runtime

class LoopbackArbRuntime(Runtime):
    
    @staticmethod
    def main(mgr):
        import time
        import numpy as np
        
        from acadia.system import Acadia, StreamConfiguration
        from acadia.channel import Channel
        from acadia.arrays import ProceduralWaveform
        from acadia.data import DataManager, ArrayRecordGroup
        
        acadia = Acadia()

        pulse_channel = acadia.DAC(1)

        def pulse_shape(out, sample_times):
            out[:] = Channel.to_samples(0.99*np.ones(len(sample_times), dtype=np.complex64))

        pulse = ProceduralWaveform(pulse_channel, generator=pulse_shape, region=pulse_channel)

        capture_channel = acadia.ADC(1)
        capture_data = ProceduralWaveform(capture_channel, generator=None, length=5000e-9, region=acadia.PLDDR0Array)
        capture_configuration = StreamConfiguration(capture_channel, acadia=acadia)

        def configure():
            pulse_channel.set_nyquist_zone(2)
            pulse_channel.configure_nco(frequency=2000e6)
            pulse_channel.set_vop(20000)
            
            capture_channel.set_nyquist_zone(2)
            capture_channel.set_dsa(0)
            
        SHOTS = 1000
            
        # We'll collect the data traces in a record group
        traces = ArrayRecordGroup((len(capture_data),), 
                                SHOTS, 
                                dtype=np.complex64, 
                                record_axes=[capture_data.axis()])
            
        # Make a data manager for storing data and serving it to a plotter

        mgr.add_group("traces", traces)
        
        # Create a sequence for the sequencer
        def sequence(a):
            capture_configuration.reset()
            for i in range(100):
                a._active_sequencer.nop()
                
            # with a.channel_synchronizer():
            #     a.generate(pulse)
            #     a.capture(capture_configuration, capture_data)
            
            with a.channel_synchronizer(block=False):
                a.capture(capture_configuration, capture_data)
                a.generate(pulse)
                a.generate(pulse)
                a.generate(pulse)
                
            # pulses = a.sequencer().DSP()
            # pulses.load(3)
                
            # with a.sequencer().repeat_until(pulses == 5):
            #     with a.sequencer().test(a.all_channel_fifos_empty(pulse_channel)):
            #         with a.channel_synchronizer(trigger=False, block=False):
            #             a.generate(pulse)
            #         pulses += 1

        # Because the pulse is procedurally generated, we can change its length at runtime,
        # so we need to start by allocating the length to use initially
        pulse.allocate(500e-9)

        # Attach to the hardware
        acadia.attach()

        # Load the wave memory with the pulse by calling the generator function
        pulse.populate()

        # Configure channel parameters using the function we defined above
        configure()

        # Configure the stream processing path to capture data using the configuration
        # written above
        acadia.configure_stream(capture_configuration)

        # Compile only once
        acadia.compile(sequence)
        
        # import logging
        # logging.basicConfig(format="[%(asctime)s] (%(threadName)s) %(levelname)s: %(message)s", level=logging.DEBUG)
        
        mgr.start_server()
        
        for shot in mgr.progress(range(SHOTS)):
            acadia.run(assemble=(shot==0))
            
            # Get the trace data and write it into a record
            trace = Channel.from_samples(capture_data.memory())
            mgr.append("traces", trace)
            
            # # Wait some time until running again just so that we can see the plot update
            # time.sleep(0.001)
                    
    # def plot(self):
    #     import matplotlib.pyplot as plt
    #     import time
    #     import numpy as np
        
    #     from acadia.data import DataManager
        
    #     fig,ax = plt.subplots(figsize=(8,4))
    #     (line_re,) = ax.plot([], [])
    #     (line_im,) = ax.plot([], [])
    #     ax.set_ylim(-0.01, 0.01)
    #     ax.set_xlim(0,5)
    #     ax.grid()
        
    #     def update(address, frame):
    #         traces = DataManager.receive_group("traces", address)
            
    #         if traces is not None:
    #             # Just plot only the most recently received trace
    #             axis = traces.axis()*1e6
    #             trace = traces.data()[-1,:]
                
    #             # Prepare the background
    #             line_re.set_data(axis, np.real(trace))
    #             line_im.set_data(axis, np.imag(trace))
    #             # ax.set_title(str( frame))
                
    #             # ax.relim()
    #             # ax.autoscale_view()
    #         return line_re, line_im
        
    #     return fig, update

Overwriting loopback_arb.py


In [2]:
%matplotlib widget
from loopback_arb import LoopbackArbRuntime
import logging

rt = LoopbackArbRuntime("192.168.2.69", "loopback_arb.py")
rt.run()

In [3]:
print(rt._proc.stdout.read().decode())

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/home/root/acadia/pyacadia/acadia/runtime.py", line 178, in remote_main
    raise exc
  File "/home/root/acadia/pyacadia/acadia/runtime.py", line 158, in remote_main
    cls.main(mgr)
TypeError: main() takes 0 positional arguments but 1 was given



In [4]:
print(rt._proc.stdout.read().decode("ascii"))

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/home/root/acadia/pyacadia/acadia/runtime.py", line 145, in remote_main
    mgr.start_server()
  File "/home/root/acadia/pyacadia/acadia/data.py", line 667, in start_server
    preexec_fn=os.setsid)
  File "/usr/lib/python3.7/subprocess.py", line 800, in __init__
    restore_signals, start_new_session)
  File "/usr/lib/python3.7/subprocess.py", line 1551, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'python3 -c "import logging; logging.basicConfig(filename=\\"/home/root/data_server.log\\", level=logging.DEBUG, filemode=\\"w\\"); from acadia.data import DataManager; DataManager._server_process_func(\\"/home/root/101223-193903\\", (\\"\\", 6672)); ': 'python3 -c "import logging; logging.basicConfig(filename=\\"/home/root/data_server.log\\", level=logging.DEBUG, filemode=\\"w\\"); from acadia.data import DataManager

In [5]:
rt.stop()

ConnectionRefusedError: [Errno 111] Connection refused